In [6]:
import fastf1
from fastf1 import Cache

Cache.enable_cache('cache')
Cache.enable_cache(
    cache_dir='cache',
    ignore_version=False,
    force_renew=False,
    use_requests_cache=True
)
schedule = fastf1.get_event_schedule(2025)
schedule = schedule[['RoundNumber', 'EventDate', 'EventName']]
print(schedule)

    RoundNumber  EventDate                  EventName
0             0 2025-02-28         Pre-Season Testing
1             1 2025-03-16      Australian Grand Prix
2             2 2025-03-23         Chinese Grand Prix
3             3 2025-04-06        Japanese Grand Prix
4             4 2025-04-13         Bahrain Grand Prix
5             5 2025-04-20   Saudi Arabian Grand Prix
6             6 2025-05-04           Miami Grand Prix
7             7 2025-05-18  Emilia Romagna Grand Prix
8             8 2025-05-25          Monaco Grand Prix
9             9 2025-06-01         Spanish Grand Prix
10           10 2025-06-15        Canadian Grand Prix
11           11 2025-06-29        Austrian Grand Prix
12           12 2025-07-06         British Grand Prix
13           13 2025-07-27         Belgian Grand Prix
14           14 2025-08-03       Hungarian Grand Prix
15           15 2025-08-31           Dutch Grand Prix
16           16 2025-09-07         Italian Grand Prix
17           17 2025-09-21  

In [7]:
import pandas as pd
import numpy as np


def build_season(year):
    schedule = fastf1.get_event_schedule(year)
    schedule = schedule[schedule['RoundNumber'] > 0]  

    season_rows = []
    for rnd in schedule['RoundNumber']:
        rnd = int(rnd)
        event_name = schedule.loc[schedule['RoundNumber'] == rnd, 'EventName'].values[0]
        try:
            session = fastf1.get_session(year, rnd, 'R')
            session.load(laps=True, weather=True)
        except Exception as e:
            print(f"Skipping {year} round {rnd} ({event_name}): {e}")
            continue

        # Race results already carry starting grid + finishing position
        df = session.results.copy()
        df['Year'] = year
        df['Round'] = rnd
        df['EventName'] = event_name
        df = df.rename(columns={
            'GridPosition': 'StartingGridPosition',
            'Position': 'FinishingPosition',
        })

        # --- Lap time per driver (seconds) ---
        laps = session.laps
        if laps is not None and not laps.empty:
            lap_agg = laps.groupby('Driver')['LapTime'].agg(
                AvgLapTime='mean',
                MedianLapTime='median',
                FastestLapTime='min',
            ).reset_index()
            for col in ['AvgLapTime', 'MedianLapTime', 'FastestLapTime']:
                lap_agg[col] = lap_agg[col].dt.total_seconds()
            df = df.merge(lap_agg, left_on='Abbreviation', right_on='Driver', how='left')
            df = df.drop(columns=['Driver'])

        # --- Weather (session-level; same value for every driver at this track) ---
        weather = session.weather_data
        if weather is not None and not weather.empty:
            df['AirTemp'] = weather['AirTemp'].mean()
            df['TrackTemp'] = weather['TrackTemp'].mean()
            df['Humidity'] = weather['Humidity'].mean()
            df['Pressure'] = weather['Pressure'].mean()
            df['WindSpeed'] = weather['WindSpeed'].mean()
            df['Rainfall'] = bool(weather['Rainfall'].any())

        season_rows.append(df)

    season_df = pd.concat(season_rows, ignore_index=True)

    keep = [
        'Year', 'Round', 'EventName',
        'Abbreviation', 'DriverNumber', 'TeamName',
        'StartingGridPosition', 'FinishingPosition',
        'AvgLapTime', 'MedianLapTime', 'FastestLapTime',
        'AirTemp', 'TrackTemp', 'Humidity', 'Pressure', 'WindSpeed', 'Rainfall',
    ]
    keep = [c for c in keep if c in season_df.columns]
    return season_df[keep]


In [8]:
# Build 2024: weather, lap time, starting grid + finishing position per driver per track
results_2024_df = build_season(2024)
print(results_2024_df.shape)
results_2024_df.head(20)


core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No ca

(479, 17)


,Year,Round,EventName,Abbreviation,DriverNumber,TeamName,StartingGridPosition,FinishingPosition,AvgLapTime,MedianLapTime,FastestLapTime,AirTemp,TrackTemp,Humidity,Pressure,WindSpeed,Rainfall
0,2024,1,Bahrain Grand Prix,VER,1,Red Bull Racing,1.0,1.0,96.574421,95.6790,92.608,18.227389,23.652866,48.821656,1017.185987,0.785987,False
1,2024,1,Bahrain Grand Prix,PER,11,Red Bull Racing,5.0,2.0,96.968404,96.2490,94.364,18.227389,23.652866,48.821656,1017.185987,0.785987,False
2,2024,1,Bahrain Grand Prix,SAI,55,Ferrari,4.0,3.0,97.014947,96.2200,94.507,18.227389,23.652866,48.821656,1017.185987,0.785987,False
3,2024,1,Bahrain Grand Prix,LEC,16,Ferrari,2.0,4.0,97.270368,96.7960,94.090,18.227389,23.652866,48.821656,1017.185987,0.785987,False
4,2024,1,Bahrain Grand Prix,RUS,63,Mercedes,3.0,5.0,97.395263,96.6830,95.065,18.227389,23.652866,48.821656,1017.185987,0.785987,False
5,2024,1,Bahrain Grand Prix,NOR,4,McLaren,7.0,6.0,97.424561,96.6200,94.476,18.227389,23.652866,48.821656,1017.185987,0.785987,False
6,2024,1,Bahrain Grand Prix,HAM,44,Mercedes,9.0,7.0,97.457298,96.6940,94.722,18.227389,23.652866,48.821656,1017.185987,0.785987,False
7,2024,1,Bahrain Grand Prix,PIA,81,McLaren,8.0,8.0,97.558316,96.7960,94.774,18.227389,23.652866,48.821656,1017.185987,0.785987,False
8,2024,1,Bahrain Grand Prix,ALO,14,Aston Martin,6.0,9.0,97.888228,97.2650,94.199,18.227389,23.652866,48.821656,1017.185987,0.785987,False
9,2024,1,Bahrain Grand Prix,STR,18,Aston Martin,12.0,10.0,98.209789,97.1750,95.632,18.227389,23.652866,48.821656,1017.185987,0.785987,False


In [9]:
# Build 2025: weather, lap time, starting grid + finishing position per driver per track
results_2025_df = build_season(2025)
print(results_2025_df.shape)
results_2025_df.head(20)


core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No

Skipping 2025 round 15 (Dutch Grand Prix): any API: 500 calls/h


core           INFO 	Loading data for Azerbaijan Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


Skipping 2025 round 16 (Italian Grand Prix): any API: 500 calls/h


core           INFO 	Loading data for Singapore Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


...

core           INFO 	Loading data for Qatar Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


Skipping 2025 round 22 (Las Vegas Grand Prix): any API: 500 calls/h


core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


Skipping 2025 round 23 (Qatar Grand Prix): any API: 500 calls/h
Skipping 2025 round 24 (Abu Dhabi Grand Prix): any API: 500 calls/h
(279, 17)


,Year,Round,EventName,Abbreviation,DriverNumber,TeamName,StartingGridPosition,FinishingPosition,AvgLapTime,MedianLapTime,FastestLapTime,AirTemp,TrackTemp,Humidity,Pressure,WindSpeed,Rainfall
0,2025,1,Australian Grand Prix,NOR,4,McLaren,1.0,1.0,103.428302,90.551,82.167,15.707865,18.942135,78.421348,1009.901685,3.475281,True
1,2025,1,Australian Grand Prix,VER,1,Red Bull Racing,3.0,2.0,103.341151,91.271,83.081,15.707865,18.942135,78.421348,1009.901685,3.475281,True
2,2025,1,Australian Grand Prix,RUS,63,Mercedes,4.0,3.0,103.686340,91.856,85.065,15.707865,18.942135,78.421348,1009.901685,3.475281,True
3,2025,1,Australian Grand Prix,ANT,12,Mercedes,16.0,4.0,104.579370,93.697,84.901,15.707865,18.942135,78.421348,1009.901685,3.475281,True
4,2025,1,Australian Grand Prix,ALB,23,Williams,6.0,5.0,104.672389,93.051,84.597,15.707865,18.942135,78.421348,1009.901685,3.475281,True
5,2025,1,Australian Grand Prix,STR,18,Aston Martin,13.0,6.0,104.730130,93.631,85.538,15.707865,18.942135,78.421348,1009.901685,3.475281,True
6,2025,1,Australian Grand Prix,HUL,27,Kick Sauber,17.0,7.0,104.740167,94.064,85.243,15.707865,18.942135,78.421348,1009.901685,3.475281,True
7,2025,1,Australian Grand Prix,LEC,16,Ferrari,7.0,8.0,103.933528,92.448,85.271,15.707865,18.942135,78.421348,1009.901685,3.475281,True
8,2025,1,Australian Grand Prix,PIA,81,McLaren,2.0,9.0,102.084462,90.722,83.242,15.707865,18.942135,78.421348,1009.901685,3.475281,True
9,2025,1,Australian Grand Prix,HAM,44,Ferrari,8.0,10.0,103.967189,92.914,84.218,15.707865,18.942135,78.421348,1009.901685,3.475281,True


In [10]:
import openpyxl as xl
results_2024_df.to_excel('f1_2024_results.xlsx', index=False)
#raw data

In [11]:
results_2025_df.to_excel('f1_2025_results.xlsx', index=False)

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

feature_df = pd.DataFrame()

encoder_driver_id = LabelEncoder()
encoder_event = LabelEncoder()
encoder_team = LabelEncoder()
encoder_rain = LabelEncoder()

imputer = SimpleImputer(strategy='median')
lap_cols = ['AvgLapTime', 'MedianLapTime', 'FastestLapTime']
imputer.fit(pd.concat([results_2024_df[lap_cols], results_2025_df[lap_cols]]))

all_drivers = results_2024_df['Abbreviation']
all_events = results_2024_df['EventName']
all_teams = results_2024_df['TeamName']
all_rain = results_2024_df['Rainfall'] 

feature_df['driver_id'] = encoder_driver_id.fit_transform(all_drivers)
feature_df['event_id'] = encoder_event.fit_transform(all_events)
feature_df['team_id'] = encoder_team.fit_transform(all_teams)
feature_df['rainfall'] = encoder_rain.fit_transform(all_rain)

num_cols = ['StartingGridPosition', 'AvgLapTime', 'MedianLapTime', 'FastestLapTime',
            'AirTemp', 'TrackTemp', 'Humidity', 'Pressure', 'WindSpeed']

scaler = StandardScaler()
scaler.fit(results_2024_df[num_cols])

X_train = scaler.transform(results_2024_df[num_cols])
X_test = scaler.transform(results_2025_df[num_cols])

Y_train = results_2024_df['FinishingPosition']
Y_test = results_2025_df['FinishingPosition']




c:\Users\Paul Abruzzo\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


ValueError: Expected a 2-dimensional container but got <class 'pandas.core.series.Series'> instead. Pass a DataFrame containing a single row (i.e. single sample) or a single column (i.e. single feature) instead.

In [ ]:
model = GradientBoostingRegressor()
model.fit(X_train, Y_train)

# --- AFTER TRAINING ---
# The data has been converted into these specific parameters:
print("--- Learned Parameters ---")
print(f"Weights (model.coef_):      {model.coef_}")
print(f"Intercept (model.intercept_): {model.intercept_:.2f}")